<a href="https://colab.research.google.com/github/anuradha-gh/AI-Powered-Granular-Access-Control-for-SaaS-Applications-/blob/Role-Recommendation/Role_Recommendation_System.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

# @title 1. Setup and Installations
!pip install transformers[torch] datasets scikit-learn -q

import json
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from datasets import Dataset, DatasetDict
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)
from google.colab import drive

print("Libraries installed and imported.")

# @title 2. Load and Parse CloudTrail Log Data
# --- Configuration ---
FILE_PATH = '/content/drive/MyDrive/flaws_cloudtrail00.json'
# ---------------------

try:
    drive.mount('/content/drive')
    with open(FILE_PATH, 'r') as f:
        log_data = json.load(f)
    records = log_data.get('Records', [])
    df = pd.json_normalize(records)
    print(f"Successfully loaded {len(df)} log records from {FILE_PATH}.")
except FileNotFoundError:
    print(f"ERROR: File '{FILE_PATH}' not found.")
    df = pd.DataFrame()

# @title 3. Log-to-Sentence Transformation (Blueprint Sec 1.2.1)
def log_to_sentence(row):
    """
    Transforms a log entry (DataFrame row) into a natural language sentence
    that the BERT model can understand.
    """
    event_time = row.get('eventTime', 'an unknown time')
    ip = row.get('sourceIPAddress', 'an unknown IP address')
    event_name = row.get('eventName', 'an unknown action')
    event_source = row.get('eventSource', 'an unknown source')
    user_name = row.get('userIdentity.userName', 'an unknown user')
    user_type = row.get('userIdentity.type', 'an unknown identity type')
    region = row.get('awsRegion', 'an unknown region')
    user_name = re.sub(r'[^a-zA-Z0-9_.-]', '', str(user_name)) if user_name else 'unknown_user'

    sentence = (
        f"At {event_time}, a user identified as '{user_name}' of type '{user_type}' "
        f"from IP address {ip} performed the action '{event_name}' on the service "
        f"'{event_source}' in the '{region}' region."
    )
    return sentence

if not df.empty:
    df['sentence'] = df.apply(log_to_sentence, axis=1)
    print("Successfully transformed log entries into sentences.")
    print("\nExample Sentence:")
    print(df['sentence'].iloc[0])


# @title 4. Synthetic Role Label Generation
# NOTE: This is a placeholder for the demo. In a real scenario, this
# data would be labeled by the customer's security team.
def assign_synthetic_role(username):
    if not isinstance(username, str): return 'Unknown'
    username_lower = username.lower()
    if 'root' in username_lower: return 'RootAdmin'
    elif 'admin' in username_lower: return 'Administrator'
    elif 'backup' in username_lower: return 'BackupOperator'
    elif 'dev' in username_lower: return 'Developer'
    elif 'level' in username_lower: return 'SupportEngineer'
    else: return 'StandardUser'

if not df.empty:
    df['optimal_role'] = df['userIdentity.userName'].apply(assign_synthetic_role)

    # Encode labels into integers
    label_encoder = LabelEncoder()
    df['label'] = label_encoder.fit_transform(df['optimal_role'])

    id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
    label2id = {label: i for i, label in id2label.items()}
    num_labels = len(id2label)

    print(f"Generated {num_labels} synthetic roles: {list(id2label.values())}")


# @title 5. Load Model and Prepare Tokenizer
if not df.empty:
    # Load tokenizer and model
    model_checkpoint = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(model_checkpoint)

    # Load the model with a classification head
    # The weights for this head are *uninitialized*
    model = AutoModelForSequenceClassification.from_pretrained(
        model_checkpoint,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id
    )

    print(f"Successfully loaded pre-trained model: {model_checkpoint}")
    print("The model's classification head is now randomly initialized.")

    # Tokenization function
    def tokenize_function(examples):
        return tokenizer(examples["sentence"], padding="max_length", truncation=True)

    print("\n--- Tokenizer Demo ---")
    example_sentence = df['sentence'].iloc[0]
    print(f"Original sentence: \n{example_sentence}")

    tokenized_example = tokenizer(example_sentence)
    print(f"\nTokenized IDs: \n{tokenized_example['input_ids']}")

    # Convert a small sample to a Hugging Face Dataset for the demo
    demo_df = df.sample(n=100)[['sentence', 'label']]
    hg_demo_dataset = Dataset.from_pandas(demo_df)

    tokenized_demo_dataset = hg_demo_dataset.map(tokenize_function, batched=True)
    print(f"\nSuccessfully tokenized a demo dataset of {len(tokenized_demo_dataset)} records.")
    print(tokenized_demo_dataset)


# @title 6. Show Results and Next Steps
if 'model' in locals():


SyntaxError: incomplete input (ipython-input-4156500004.py, line 132)